# Session 4 - What the model knows, and where it stops working

**Student notebook.** Cells marked **EXERCISE** are for you to fill in. Everything else can be run as it stands.

## What the model knows, and where it stops working

A model with a defensible cross-validated score can still be useless in
deployment, and the reason is almost never the score. Today: whether the signal
is real at all, what the model is actually using, and the question that matters
most for a QSAR model in practice — **which molecules is this prediction valid
for?**

**This session (2 h)**

1. y-scrambling: an empirical null for your pipeline
2. Residuals: where the model fails, and why these descriptors have a floor
3. Feature importance, and why two methods disagree
4. The applicability domain: what happens outside the training chemistry
5. Capstone, to hand in

In [ ]:
import sys, warnings
sys.path.insert(0, "..")          # so that `import qsar_helpers` works
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qsar_helpers import (load_fish, load_aquatic, xy, fold_error,
                          molecule_groups, score_table,
                          DESCRIPTOR_NOTES, TARGET_LABEL)

RNG = 0                            # one seed for the whole notebook
plt.rcParams.update({"figure.figsize": (6, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
print("pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (KFold, RepeatedKFold, train_test_split,
                                     cross_val_score, cross_val_predict)
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, root_mean_squared_error

fish = load_fish()
X, y = xy(fish)
cv5 = KFold(5, shuffle=True, random_state=RNG)

## 1. Is there any signal? (y-scrambling)

The standard sanity check in QSAR, and a good habit everywhere: shuffle the
target, leaving the descriptor matrix untouched, and run the **entire** pipeline.
Any structure the procedure finds in shuffled data is structure it invented.

### EXERCISE 4.1

Cross-validate the random forest on the real target. Then build a null
distribution from 30 shuffles of `y`, plot it with the real score marked, and
report where the real score falls relative to the null.

*Runtime about two minutes.*

In [ ]:
# TODO: real CV R2; then 30 shuffles of y, same CV, collect the null; plot both
...

The null sits at **-0.19 ± 0.04** and never gets above -0.09, while the real
score is +0.60. The signal is unambiguous.

Note where the null is centred: not at zero, but clearly *below* it. A flexible
model fitted to noise is actively worse than predicting the mean on held-out
data. That is why "my R<sup>2</sup> is positive, so there must be signal" is not
an argument — the comparison you need is against this distribution, and when
leakage is present the null moves up with the real score (try 4.1 again with the
feature-selection mistake from session 2 if you want to see it).

## 2. Residuals

### EXERCISE 4.2

Use `cross_val_predict` to get an out-of-fold prediction for every molecule.
Then:

1. Plot predicted against observed, with the 1:1 line.
2. Plot the residual against `MLOGP`. Is the error uniform across lipophilicity?
3. List the five molecules with the largest absolute residual, with their
   descriptor values. Any pattern?
4. Among the rows that share an identical descriptor vector with another row
   (session 2), compute the spread in measured LC50. What does this imply about
   the best RMSE any model on these six descriptors could achieve?

In [ ]:
# TODO: out-of-fold predictions; parity plot; residual vs MLOGP;
#       worst five molecules; LC50 spread within identical descriptor vectors
...

Two things come out of this.

The residuals are not uniform, but not in the direction most people guess. RMSE
by `MLOGP` tertile is **1.03 (low) / 0.72 (mid) / 0.95 (high)** — the model does
*worst* on the least lipophilic third. And the five largest residuals are nearly
all large **under-predictions**: compounds measured as highly toxic (LC50 of 8.1,
9.6, 6.4, 7.8 on the -log10 scale) whose `MLOGP` is low or moderate, predicted
near 3-5.

That is a recognisable pattern in ecotoxicology rather than a modelling
artefact. Baseline narcosis sets a floor on toxicity as a function of
lipophilicity; compounds that act by a *specific* mechanism — binding a receptor
or inhibiting an enzyme — are far more toxic than their log P predicts, which
the field calls **excess toxicity**. A model whose dominant descriptor is `MLOGP`
reproduces the baseline and necessarily misses those compounds, and six bulk
descriptors carry nothing that would flag them. Worth asking the room: in which
direction does this model fail, and is that the safe direction for a screening
tool?

More fundamentally: rows with **identical descriptor vectors** differ in measured
LC50 by 0.32 log units on average, and by up to 3.1 log units in the worst case.
No model built on these six descriptors can distinguish those molecules — they
are literally the same input. That within-group spread is an irreducible floor
on achievable RMSE, and our model's RMSE of 0.91 is well above it, so the
descriptors are not yet the binding constraint. But it does bound what any
amount of modelling on this representation could deliver, which is the kind of
statement worth making before anyone proposes a neural network.

## 3. Feature importance, and two methods that disagree

### EXERCISE 4.3

Fit a random forest on 75% of the data. Compute

- the built-in `feature_importances_` (mean impurity decrease, computed on the
  **training** data), and
- `permutation_importance` on the **held-out** 25%, with 30 repeats.

Plot them side by side. Where do they disagree, and which would you trust for
the claim "descriptor X matters for toxicity"?

In [ ]:
# TODO: fit RF on a 75/25 split; impurity importance vs permutation importance
#       on the held-out part; plot both
...

`MLOGP` dominates both rankings, which is what the narcosis mechanism predicts.
But the two methods disagree on the rest: impurity importance gives `NdsCH`
0.03 while permuting it on held-out data costs 0.09 of R<sup>2</sup>, and the
ordering of the middle features changes.

They measure different things. Impurity importance is computed on the training
data and is biased toward features with many distinct values — a feature with
500 unique values offers far more candidate split points than a 0-4 count, so it
accumulates impurity decrease even when it is partly fitting noise. Permutation
importance on held-out data asks a question you actually care about: how much
predictive performance is lost if this column is destroyed?

### EXERCISE 4.4 — correlated descriptors break both

Now switch to the aquatic dataset (`load_aquatic()`), where `TPSA(Tot)` and
`SAacc` are correlated at r = 0.86 — both are measures of polar surface.

1. Compute both importances there. Look at `TPSA(Tot)`, `SAacc` and `GATS1p`.
2. Drop `SAacc` entirely, refit, and re-score. What happens to the model's
   performance, and what happens to `TPSA(Tot)`'s importance?
3. Write one sentence stating what a permutation importance of ~0 means when the
   feature has a correlated partner in the matrix.

In [ ]:
aqua = load_aquatic()
Xa, ya = xy(aqua)
print("corr(TPSA, SAacc) =", round(Xa[["TPSA(Tot)", "SAacc"]].corr().iloc[0, 1], 3))

# TODO: (1) impurity + permutation importance on the aquatic data
# TODO: (2) drop SAacc, refit, compare CV R2 and TPSA importance
...

Dropping a descriptor correlated at 0.86 with another leaves cross-validated
performance essentially unchanged — 0.512 with all eight descriptors, 0.501
without `SAacc`, a difference well inside the ±0.08 fold-to-fold spread — while
the importances rearrange. `H-050` goes from 0.029 to 0.089, and `TPSA(Tot)`
from 0.217 to 0.190: the model redistributes what it leans on, and the
redistribution does not even go to the dropped descriptor's closest partner.

So feature importance is a statement about **this fitted model on this
descriptor set**, not about mechanism. If the question is "does polar surface
area matter for *Daphnia* toxicity", importance rankings are not the instrument
that answers it.

## 4. The applicability domain

Cross-validation answers: how well does the model predict molecules *drawn from
the same pool* as the training molecules? That is rarely the deployment question.
In practice the model is used on a new chemical series — compounds more
lipophilic, larger, or differently substituted than anything in the training set.

### EXERCISE 4.5

Simulate that honestly on both datasets. Take the top quartile of `MLOGP` as the
test set and train on the remaining 75%. Compare with a random split of the
same size.

1. Do this for the fish dataset, reporting R<sup>2</sup> and RMSE for both splits.
2. Repeat for the aquatic dataset.
3. Compare the mean LC50 of the training and test portions. What systematic
   error do you expect, and in which direction?

In [ ]:
# TODO: for each dataset, hold out the top MLOGP quartile, train on the rest,
#       and compare with random splits of the same size
...

This is the largest effect in the whole course. On the fish data,
cross-validated R<sup>2</sup> of 0.61 becomes **0.04** when the test molecules are
simply more lipophilic than the training molecules. On the aquatic data it
becomes **-0.66** — markedly worse than predicting the training mean.

The mechanism is visible in the parity plot: the forest cannot predict outside
the range of target values it was trained on, because every prediction is an
average of training leaves. Mean LC50 rises from 3.62 to 5.41 across the split,
and the predictions stay near the training range, so the model is systematically
**under-predicting toxicity** — exactly the direction that matters for a safety
assessment.

Note also that the choice of splitting descriptor matters: holding out the top
quartile of `CIC0` instead gives R<sup>2</sup> = 0.28, not 0.04. "Out of domain"
is not one thing.

### EXERCISE 4.6 — an applicability-domain indicator

A model used in practice should refuse to answer, or flag its answer, for
molecules unlike its training set. Build the simplest such indicator.

1. Fit `NearestNeighbors` on the **scaled** training descriptors. For each test
   molecule compute the mean distance to its 5 nearest training molecules.
2. Split the test molecules into quartiles of that distance and report the RMSE
   in each quartile.
3. Verify the indicator also flags the high-`MLOGP` molecules from 4.5.

In [ ]:
# TODO: kNN-distance to the training set as a domain indicator;
#       RMSE by distance quartile
...

A crude indicator, and it works: error rises across the distance quartiles, and
the high-`MLOGP` molecules sit far outside the distances seen in a random
held-out sample. A deployed QSAR model should report this alongside every
prediction; the OECD validation principles require a defined applicability
domain for exactly this reason.

## 5. Capstone — to hand in

You have a second dataset you have barely used: `load_aquatic()`, 546 molecules,
eight descriptors, LC50 toward *Daphnia magna*. Treat it as a new project and
apply everything from the four sessions.

**Deliverable: one notebook plus a methods paragraph of at most 250 words.**

Required content.

1. **Baseline.** The dummy's R<sup>2</sup> and RMSE, and the RMSE as a
   fold-error in concentration.
2. **A tuned model, selected honestly.** Choose among at least three model
   families with at least three hyperparameter settings each. Report either a
   nested-CV estimate or a flat estimate explicitly labelled as an upper bound,
   and state how many configurations you tried.
3. **A spread, not a point.** Repeated cross-validation, mean ± sd, and a
   sentence on whether your chosen model is distinguishable from the runner-up.
4. **Leakage audit.** State for each data-dependent step where it sits relative
   to the resampling loop. Report the number of rows sharing an identical
   descriptor vector and what you did about it.
5. **A y-scrambling null** for your final pipeline.
6. **An applicability domain.** One extrapolation experiment of your choice, one
   domain indicator, and a statement of which molecules your model should not be
   used for.
7. **One honest negative result.** Something you expected to matter that did
   not, with the number that shows it.

The methods paragraph should be written so that a reader could reproduce your
estimate and would not be surprised by your model's performance in deployment.

Marking weights the audit and the applicability domain as heavily as the
performance. A submission reporting R<sup>2</sup> = 0.50 with a correct audit
scores above one reporting 0.60 without.

In [ ]:
# Capstone scaffold. Everything below is yours to replace.
aqua = load_aquatic()
Xa, ya = xy(aqua)
print(aqua.shape)
print(f"{len(set(molecule_groups(Xa)))} distinct descriptor vectors in {len(Xa)} rows")
aqua.describe().T.round(2)

## Reporting checklist

What a defensible supervised-learning result contains. Keep this; it applies
well beyond QSAR.

1. The baseline, in the same units and on the same folds.
2. The performance metric in units someone in the field can interpret — a
   fold-error, not only an R<sup>2</sup>.
3. An estimate with a spread, from repeated resampling, not a single split.
4. The resampling scheme, including the grouping variable if the rows are not
   independent.
5. Every data-dependent step located relative to the resampling loop.
6. The number of model configurations tried, and whether selection was nested.
7. An empirical null for the whole pipeline.
8. The applicability domain, and the performance drop outside it.
9. At least one thing you checked that turned out not to matter.